## SetFit Model Inference for Denoising

#### Load in Python Libraries

In [14]:
from setfit import SetFitModel
import pickle 
import pandas as pd
import numpy as np
from tqdm import tqdm
import mlflow 

#### Set Experiment

In [2]:
mlflow.set_experiment("adeptID")

<Experiment: artifact_location='file:///c:/Users/jvhua/OneDrive/Desktop/ISYE-CSE-MGT-6748-Group-1/02_preprocess/mlruns/418082982984917409', creation_time=1719438288344, experiment_id='418082982984917409', last_update_time=1719438288344, lifecycle_stage='active', name='adeptID', tags={}>

#### Helper Functions

In [3]:
def count_words(text):

    words = text.split()

    return len(words)

def create_dataframe(data):

    data_list = [(k, v) for k, vals in data.items() for v in vals]

    df = pd.DataFrame(data_list, columns=['uuid', 'text'])

    return df

def filter_and_merge_data(data1, labels, original_data, count_words):

    data1['pred_label'] = labels

    data1['preds'] = data1['pred_label'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)

    filtered_data = data1[data1['preds'] != 0]
    non_signal_data = data1[data1['preds'] == 0]


    data1_filtered = filtered_data.groupby('uuid')['text'].apply(' '.join).reset_index()
    data1_filtered_no_signal = non_signal_data.groupby('uuid')['text'].apply(' '.join).reset_index()


    uuid_list = data1_filtered['uuid'].tolist()

    filtered_original_data = original_data[original_data['id'].isin(uuid_list)]

    filtered_original_data['word_count_original'] = filtered_original_data['body'].apply(count_words)
    data1_filtered['word_count_signals'] = data1_filtered['text'].apply(count_words)
    data1_filtered = data1_filtered.rename(columns = {'text': 'signals_text'})
    data1_filtered_no_signal['word_count_no_signal'] = data1_filtered_no_signal['text'].apply(count_words)
    data1_filtered_no_signal = data1_filtered_no_signal.rename(columns = {'text': 'no_signals_text'})

    new_data = data1_filtered.merge(filtered_original_data, left_on = 'uuid', right_on = 'id', how = 'left')
    new_data1 = new_data.merge(data1_filtered_no_signal, left_on = 'uuid', right_on = 'uuid', how = 'left')

    
    return filtered_original_data, new_data1, data1_filtered

#### Load in Data and Model and Create DataFrame

In [4]:
original_data = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\unclassified_postings.csv", index_col =0)

##### Load in model from local Huggingface model folder

In [5]:
model_name = "setfit_model_v5"
model = SetFitModel.from_pretrained(f"C:\\Users\\jvhua\\OneDrive\\Desktop\\ISYE-CSE-MGT-6748-Group-1\\{model_name}", trust_remote_code = True)

'setfit_model_v5'

In [6]:
batch_no = 'batch11'

with open(f"C:\\Users\\jvhua\\OneDrive\Desktop\\ISYE-CSE-MGT-6748-Group-1\\02_preprocess\\job_posting_chunk_{batch_no}.pickle", 'rb') as file:
    data = pickle.load(file)

data1 = create_dataframe(data)

<>:3: DeprecationWarning: invalid escape sequence '\D'
<>:3: DeprecationWarning: invalid escape sequence '\D'
C:\Users\jvhua\AppData\Local\Temp\ipykernel_16656\1291770850.py:3: DeprecationWarning: invalid escape sequence '\D'
  with open(f"C:\\Users\\jvhua\\OneDrive\Desktop\\ISYE-CSE-MGT-6748-Group-1\\02_preprocess\\job_posting_chunk_{batch_no}.pickle", 'rb') as file:


'batch11'

#### SetFit Denoising

In [8]:
labels = [model.predict([sent]) for sent in tqdm(data1['text'])]

100%|██████████| 19260/19260 [09:18<00:00, 34.50it/s]


In [9]:
filtered_original_data, new_data, data1_filtered = filter_and_merge_data(data1, labels, original_data, count_words)

C:\Users\jvhua\AppData\Local\Temp\ipykernel_16656\3886928815.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_original_data['word_count_original'] = filtered_original_data['body'].apply(count_words)


#### Inspect DataFrame

In [10]:
new_data[['id','body', 'word_count_original', 'signals_text','word_count_signals', 
              'no_signals_text', 'word_count_no_signal']].head()

,id,body,word_count_original,signals_text,word_count_signals,no_signals_text,word_count_no_signal
0,0020b05796f9f5caf9c691aa328d02dd9f4f2ec1,"['MLee Healthcare', '4.0', 'ASCP Medical Labor...",499,"['MLee Healthcare', '4.0', 'ASCP Medical Labor...",382,"If a job has no salary data, Glassdoor display...",117.0
1,00336a9676f402f61188a9573ca54c72265af8ae,Clinical Liaison Behavioral Health Svcs OH Fre...,392,Clinical Liaison Behavioral Health Svcs OH Fre...,297,Yes No Education Do you have a Bachelor's degr...,95.0
2,004119874e5219369f494d9f432390cd6fd351c9,Discover Vanderbilt University Medical Center:...,1459,"programs in patient care, education, and resea...",755,Discover Vanderbilt University Medical Center:...,787.0
3,00964aecdcf1e7aa3cebbed17a89b14ea986d2d0,Part-Time Direct Support Professional (DSP 1)\...,495,Part-Time Direct Support Professional (DSP 1)\...,326,YAI promotes a person-centered approach by cre...,169.0
4,00aa915659d08f2611d57180242c54bdbba21541,Sr Manager Medical Science Liaison\nEmployer\n...,1250,Sr Manager Medical Science Liaison\nEmployer\n...,648,Insmed is a global biopharmaceutical company o...,660.0


#### Log Metrics and Save CSV for DSPy Information Extraction

In [17]:
with mlflow.start_run(run_name = "setfit-denoise-batch-inference"):
    mlflow.log_param('model', model_name)
    mlflow.log_param('batch', batch_no)
    mlflow.log_metric('mean_batch_original',  np.mean(filtered_original_data['word_count_original']))
    mlflow.log_metric('mean_batch_denoised',  np.mean(data1_filtered['word_count_signals']))
    mlflow.log_metric('median_batch_original',  np.median(filtered_original_data['word_count_original']))
    mlflow.log_metric('median_batch_denoised',  np.median(data1_filtered['word_count_signals']))
    mlflow.log_metric(f'number_sentences_in_{batch_no}', data1.shape[0])
    print(np.mean(filtered_original_data['word_count_original']), np.mean(data1_filtered['word_count_signals']))
    print(np.median(filtered_original_data['word_count_original']), np.median(data1_filtered['word_count_signals']))
    print(data1.shape)
    new_data[['id','body', 'word_count_original', 'signals_text','word_count_signals', 
              'no_signals_text', 'word_count_no_signal']].to_csv(f'C:\\Users\\jvhua\\OneDrive\\Desktop\\ISYE-CSE-MGT-6748-Group-1\\data\\version2_setfit_filter_data_{batch_no}.csv')
mlflow.end_run()

489.41424272818455 357.8134403209629
436.0 326.0
(19260, 4)
